In [0]:
# =========================================================================
# GOLD LAYER: fact_sales Worker Notebook
# =========================================================================

GOLD_STORAGE_PATH = "abfss://gold@aze2estorage.dfs.core.windows.net/facts/fact_sales"
TABLE_NAME = "gold.fact_sales"

print(f"🔄 Starting build for {TABLE_NAME}...")
print(f"📂 Target Storage Path: {GOLD_STORAGE_PATH}")

# 1. Read Multi-Year Silver Sales Tables and Union Them
print("📥 Reading 2015, 2016, and 2017 Silver sales partitions...")
df_sales_2015 = df_sales =spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/sales_2015")
df_sales_2016 = df_sales =spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/sales_2016")
df_sales_2017 = df_sales =spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/sales_2017")

df_sales_combined = df_sales_2015.unionByName(df_sales_2016).unionByName(df_sales_2017)
print(f"📊 Total combined sales rows: {df_sales_combined.count()}")

# 2. Read Silver Products for Pricing and Cost Metrics
df_products =spark.read.format("delta").load("abfss://silver@aze2estorage.dfs.core.windows.net/products")

# 3. Build Central Fact Table with Financial Metrics
fact_sales = (
    df_sales_combined
    .join(df_products, "product_key", "left")
    .select(
        df_sales_combined["order_number"],
        df_sales_combined["order_line_item"],
        df_sales_combined["order_date"],
        df_sales_combined["stock_date"],
        df_sales_combined["customer_key"],
        df_sales_combined["product_key"],
        df_sales_combined["territory_key"],
        df_sales_combined["order_quantity"],
        df_sales_combined["days_to_stock"],
        df_sales_combined["order_year"],
        df_sales_combined["order_month"],
        df_products["product_price"],
        df_products["product_cost"],
        
        # Computed Financial Metrics for Power BI Dashboard Reporting
        (df_sales_combined["order_quantity"] * df_products["product_price"]).alias("total_revenue"),
        (df_sales_combined["order_quantity"] * (df_products["product_price"] - df_products["product_cost"])).alias("total_profit_margin")
    )
)

# 4. Write Delta Files to Explicit Gold Storage Path
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_STORAGE_PATH)

print("💾 Successfully wrote Delta files to Gold storage path.")

# 5. Register/Update External Table in Unity Catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME}
    USING DELTA
    LOCATION '{GOLD_STORAGE_PATH}'
""")

print(f"🔗 Registered external table: {TABLE_NAME}")

# 6. Performance Optimization for Power BI DirectQuery
spark.sql(f"OPTIMIZE {TABLE_NAME} ZORDER BY (order_date, product_key, customer_key)")

print(f"⚡ Successfully optimized {TABLE_NAME} with Z-Ordering.")
print(f"🎉 {TABLE_NAME} build complete!")